# Variant N — Zero Expert Branch (Phase 3 Extension)

Tắt đóng góp nhánh expert bằng cách set `residual_scale=0` trên tất cả SageLayer sau khi load model.
Kiến trúc giữ nguyên, số tham số giữ nguyên — chỉ tắt residual contribution của expert path.
Chạy 2 seed (42, 43) × 35 epoch, cùng điều kiện Phase 2.

In [ ]:
# ===========================================================================
# Cell 1: Setup, Clone, Dataset
# ===========================================================================
!nvidia-smi

!rm -rf /content/SAGE_LITE
!git clone https://github.com/bach0823/SAGE_LITE.git /content/SAGE_LITE
%cd /content/SAGE_LITE
!git log --oneline -3

!python prepare_data/prepare_crack500.py

# Ancestor checkpoint Stage 1 (seed=42, từ Phase 2)
# Dùng last_model_b2_stage1_rng để resume faithful nếu có
# Nếu không có: train from scratch 2 stage
# Phase 3 đã dùng train_phase3_ablation.py với --seed => dùng lại
!python scripts/train_phase3_ablation.py --help 2>&1 | head -40

In [ ]:
# ===========================================================================
# Cell 2a: Thêm Variant N vào train_phase3_ablation.py
# Variant N: giữ kiến trúc canonical, nhưng set residual_scale=0 cho mọi SageLayer
# CÁCH THỰC HIỆN: monkey-patch SageLayer.forward để tắt expert contribution
# ===========================================================================
%cd /content/SAGE_LITE

# Kiểm tra script training hiện tại có hỗ trợ variant N chưa
!grep -n 'variant' scripts/train_phase3_ablation.py | head -20

In [ ]:
# ===========================================================================
# Cell 2b: Tạo script Variant N
# ===========================================================================
%cd /content/SAGE_LITE

variant_n_script = '''
"""Variant N: Canonical architecture + residual_scale=0 on all SageLayers.
Expert branch exists but contributes zero to forward output.
output = main_output + 0 * expert_output = main_output
Tham so: giong canonical (10,205,755 params), nhung expert pool khong duoc dung.
"""
import sys, os, json, shutil, hashlib, argparse, logging
import numpy as np
import torch
import yaml

sys.path.insert(0, "/content/SAGE_LITE")
sys.path.insert(0, "/content/SAGE_LITE/scripts")

from sage.components.sage_layer import SageLayer

def disable_expert_branch(model):
    """Set residual_scale=0 on all SageLayer instances. Expert branch outputs are zeroed."""
    n_disabled = 0
    for name, m in model.named_modules():
        if isinstance(m, SageLayer) and m.fusion_type == "residual":
            m.residual_scale = 0.0
            n_disabled += 1
    return n_disabled

# Parse args
parser = argparse.ArgumentParser()
parser.add_argument("--seed", type=int, default=42)
parser.add_argument("--config", default="configs/sage_b2_crack500.yaml")
parser.add_argument("--output_dir", default="/content/runs/phase3_N_seed{seed}")
args = parser.parse_args()

# Replace {seed} placeholder
output_dir = args.output_dir.replace("{seed}", str(args.seed))
os.makedirs(output_dir, exist_ok=True)

# Import training logic from canonical script
# Thay vi viet lai, ta patch train_phase3_ablation.py bang cach thay doi variant
print(f"Variant N se duoc chay thong qua train_phase3_ablation.py voi --variant N")
print(f"Neu chua co --variant N, phai them vao script.")
'''

with open("/content/SAGE_LITE/scripts/train_variant_N.py", "w") as f:
    f.write(variant_n_script)
print("Draft written. Checking train_phase3_ablation.py for variant support...")

In [ ]:
# ===========================================================================
# Cell 3: Xem train_phase3_ablation.py để biết cách thêm Variant N
# ===========================================================================
%cd /content/SAGE_LITE
!head -120 scripts/train_phase3_ablation.py

In [ ]:
# ===========================================================================
# Cell 4a: Patch train_phase3_ablation.py để thêm variant N
# Nếu script đã có --variant flag, ta chỉ cần chạy với --variant N
# Nếu chưa có, ta sẽ thêm logic disable_expert_branch sau khi tạo model
# ===========================================================================
%cd /content/SAGE_LITE

patch_code = '''
import json

# Read existing script
with open("scripts/train_phase3_ablation.py", "r") as f:
    src = f.read()

# Check if variant N already exists
if "variant_N" in src or "\"N\"" in src or "Variant N" in src:
    print("Variant N already supported in script")
else:
    print("Need to patch script to add Variant N")
    # Find where model is created after args are parsed
    # We need to: 1) accept N as variant, 2) disable expert branch after model creation
    print("Searching for model creation point...")
    if "create_b2_unet" in src:
        print("  -> Found create_b2_unet call")
    if "inject_sage" in src or "sage_injection" in src:
        print("  -> Found SAGE injection")
'''

exec(compile(patch_code, "<patch>", "exec"))

In [ ]:
# ===========================================================================
# Cell 4b: Chạy Variant N Seed 42 (nếu train_phase3_ablation hỗ trợ)
# Nếu không, dùng script thay thế bên dưới
# ===========================================================================
%cd /content/SAGE_LITE
SEED = 42

# Thử chạy với --variant N trước
!python scripts/train_phase3_ablation.py \
    --config configs/sage_b2_crack500.yaml \
    --variant N \
    --seed {SEED} \
    --output_dir /content/runs/phase3_N_seed{SEED} \
    2>&1 | tail -5

# Nếu báo lỗi unknown variant N, chạy cell 4c

In [ ]:
# ===========================================================================
# Cell 4c: Fallback - Inline Variant N script nếu train_phase3_ablation
#          không hỗ trợ variant N trực tiếp.
# Logic: clone train_crack.py flow nhưng sau create_model, gọi disable_expert_branch()
# ===========================================================================
%cd /content/SAGE_LITE

variant_n_inline = """
import sys, os, json, shutil, hashlib, argparse, logging, time, random
import numpy as np
import torch
import yaml
import cv2
import csv

sys.path.insert(0, "/content/SAGE_LITE")
sys.path.insert(0, "/content/SAGE_LITE/scripts")

SEED = int(os.environ.get("VARIANT_N_SEED", "42"))
OUTPUT_DIR = f"/content/runs/phase3_N_seed{SEED}"
CONFIG_PATH = "configs/sage_b2_crack500.yaml"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Import canonical training components
from sage.components.sage_layer import SageLayer
from sage.networks import create_b2_unet

def disable_expert_branch(model):
    n = 0
    for name, m in model.named_modules():
        if isinstance(m, SageLayer) and m.fusion_type == "residual":
            m.residual_scale = 0.0
            n += 1
    return n

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

print(f"=== Variant N, Seed {SEED} ===")
set_seed(SEED)

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)
config[\"root_dir\"] = \"/content/dataset/Crack500\"
config[\"seed\"] = SEED

device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")
print(f\"Device: {device}\")

sage_cfg = config.get(\"sage_config\", {})
model = create_b2_unet(
    num_transformer_layers=int(config.get(\"num_transformer_layers\", 4)),
    pretrained=True,
    sage_config=sage_cfg,
    p3_mode=config.get(\"p3_mode\"),
    use_plu_head=config.get(\"use_plu_head\", False),
    use_s2_gate=config.get(\"use_s2_gate\", False),
    s2_gate_kernel_size=config.get(\"s2_gate_kernel_size\", 3),
    use_s2_gate_block2=config.get(\"use_s2_gate_block2\", False),
    s2_gate_block2_kernel_size=config.get(\"s2_gate_block2_kernel_size\", 3),
    s2_gate_block2_dilation=config.get(\"s2_gate_block2_dilation\", 1),
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f\"Total params: {total_params:,}\")

n_disabled = disable_expert_branch(model)
print(f\"Disabled expert branch on {n_disabled} SageLayers (residual_scale -> 0.0)\")

# Verify: expert contribution is 0
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, 448, 448, device=device)
    out1 = model(dummy)
    # Re-enable briefly to compare
    for m in model.modules():
        if isinstance(m, SageLayer) and m.fusion_type == \"residual\":
            m.residual_scale = 0.1
    out2 = model(dummy)
    for m in model.modules():
        if isinstance(m, SageLayer) and m.fusion_type == \"residual\":
            m.residual_scale = 0.0
print(f\"Output diff (N vs canonical scale=0.1): {(out1 - out2).abs().max().item():.6f}  (should be > 0 if experts have effect)\")

print(\"\\n[Variant N setup verified. Tiếp tục training với train_crack.py...]\")  
"""

with open("/content/SAGE_LITE/scripts/setup_variant_N.py", "w") as f:
    f.write(variant_n_inline)
print("Written setup_variant_N.py")

In [ ]:
# ===========================================================================
# Cell 5a: Training Variant N Seed 42
# Dùng train_phase3_ablation.py hoặc wrap train_crack.py với --model-init-hook
# Vì không có --variant N, dùng train_crack.py với biến môi trường
# ===========================================================================
%cd /content/SAGE_LITE
import os
SEED = 42

# Chạy với train_phase3_ablation.py (nếu hỗ trợ)
# Nếu không: chỉnh sửa script thêm variant N (xem bên dưới)
!VARIANT_N_SEED={SEED} python scripts/train_phase3_ablation.py \
    --config configs/sage_b2_crack500.yaml \
    --variant N \
    --seed {SEED} \
    --output_dir /content/runs/phase3_N_seed{SEED} 2>&1 | tee /content/runs/phase3_N_seed{SEED}/train.log

In [ ]:
# ===========================================================================
# Cell 5b: Training Variant N Seed 43
# ===========================================================================
%cd /content/SAGE_LITE
SEED = 43
!python scripts/train_phase3_ablation.py \
    --config configs/sage_b2_crack500.yaml \
    --variant N \
    --seed {SEED} \
    --output_dir /content/runs/phase3_N_seed{SEED} 2>&1 | tee /content/runs/phase3_N_seed{SEED}/train.log

In [ ]:
# ===========================================================================
# Cell 6: Post-training — Chẩn đoán, Zip, Download
# ===========================================================================
%cd /content/SAGE_LITE
from google.colab import files
import shutil, os, json

results = {}
for seed in [42, 43]:
    run_dir = f"/content/runs/phase3_N_seed{seed}"
    # Routing diagnostics
    !python tools/analyze_routing.py \
        --config configs/sage_b2_crack500.yaml \
        --checkpoint {run_dir}/best_model_b2_global.pth \
        --output_dir {run_dir}/routing_diagnostics/full_val \
        --split val \
        --data_root /content/dataset/Crack500 \
        --variant N
    # Read val dice from metadata
    meta_path = f"{run_dir}/run_metadata.json"
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        results[f"seed{seed}"] = meta.get("val_dice", "N/A")
        print(f"Seed {seed} Val Dice: {results[f'seed{seed}']}")

# Summary
dices = [v for v in results.values() if isinstance(v, float)]
if dices:
    mu = sum(dices) / len(dices)
    print(f"\nVariant N mean Val Dice: {mu:.6f}")
    print(f"Delta vs Canonical (0.778638): {mu - 0.778638:+.6f}")
    print(f"MDE = 0.010804 -> Vuot MDE: {abs(mu - 0.778638) > 0.010804}")
    results["mean"] = mu
    results["delta_vs_canonical"] = mu - 0.778638
    results["exceeds_mde"] = abs(mu - 0.778638) > 0.010804

with open("/content/phase3_N_summary.json", "w") as f:
    json.dump(results, f, indent=2)

# Zip and download
for seed in [42, 43]:
    run_dir = f"/content/runs/phase3_N_seed{seed}"
    shutil.make_archive(f"/content/phase3_variantN_seed{seed}_full", "zip", "/content/runs", f"phase3_N_seed{seed}")
    files.download(f"/content/phase3_variantN_seed{seed}_full.zip")

files.download("/content/phase3_N_summary.json")